# Packages Installation

In [ ]:
!pip uninstall numpy -y
!pip install "numpy<2"

In [ ]:
!pip install wfdb pandas scikit-learn tensorflow matplotlib

# Import Dataset

In [28]:
import os
import pandas as pd
import wfdb
import numpy as np
import ast

# Base folder of unzipped data
BASE_PATH = "dataset"

# Load PTB-XL metadata
df = pd.read_csv(os.path.join(BASE_PATH, "ptbxl_database.csv"))
scp_df = pd.read_csv(os.path.join(BASE_PATH, "scp_statements.csv"), index_col=0)

# Load ECG Signals

In [29]:
# Filter diagnostic SCP codes
scp_df = scp_df[scp_df['diagnostic'] == 1]
df.scp_codes = df.scp_codes.apply(lambda x: ast.literal_eval(x))

# Map SCP codes to subclass names
def diagnostic_class(scp):
    scp_df
    res = set()
    for k in scp.keys():
        if k in scp_df.index:
            res.add(scp_df.loc[k].diagnostic_class)
    return list(res)
                    
df['scp_classes'] = df.scp_codes.apply(diagnostic_class)


# Keep only NORM, MI, and HYP subclasses
selected_classes = ['NORM', 'MI', 'HYP']
filtered_df = df[df['scp_classes'].isin(selected_classes)].copy()

# Assign binary labels: 0 for NORM, 1 for MI or HYP
filtered_df['label'] = filtered_df['scp_classes'].apply(lambda x: 0 if x == 'NORM' else 1)

In [31]:
df['scp_classes'].dtype

dtype('O')

In [13]:
def load_ptbxl_signal(filepath):
    record = wfdb.rdrecord(filepath)
    return record.p_signal, record.sig_name  # Shape: (time_steps, 12) for 12-lead

# Load signals
signals = []
ids = []
labels = []
leads_names = []

for idx, row in filtered_df.iterrows():
    ecg_path = row['filename_hr']
    path = os.path.join(BASE_PATH, ecg_path)
    
    try:
        sig, lead_name = load_ptbxl_signal(path)
        signals.append(sig)
        leads_names.append(lead_name)
        ids.append(row['ecg_id'])
        labels.append(row['label'])
    except Exception as e:
        print(f"Error loading {path}: {e}")

In [14]:
X = np.array(signals)  # shape: (n_samples, 5000, 12)
y = np.array(labels)

print("Loaded ECG signals shape:", X.shape)
print("Binary labels shape:", y.shape)
print("Label counts:", np.unique(y, return_counts=True))

Loaded ECG signals shape: (16843, 5000, 12)
Binary labels shape: (16843,)
Label counts: (array([0, 1]), array([9528, 7315], dtype=int64))


# Data Preprocessing

In [ ]:
from scipy.signal import butter, filtfilt
import numpy as np

def highpass_filter(signal, fs=500, cutoff=0.5, order=3):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high')
    return filtfilt(b, a, signal, axis=0)

def normalize_signal(signal):
    return (signal - np.mean(signal, axis=0)) / np.std(signal, axis=0)

def preprocess_signal(signal):
    signal = highpass_filter(signal)
    signal = normalize_signal(signal)
    return signal

In [ ]:
preprocessed_signals = normalize_signal(signals)

# Plot Data

In [ ]:
import matplotlib.pyplot as plt

for j in range(4):
    plt.figure(figsize=(15, 8))
    # Plot first 6 leads
    for i in range(6):
        plt.subplot(6, 1, i + 1)
        # plt.plot(signals[j][:, i])
        plt.plot(preprocessed_signals[j][:, i])
        plt.title(f"Lead {leads_names[j][i]}")
        plt.xlabel("Time")
        plt.ylabel("Amplitude")

    plt.tight_layout()
    plt.show()